# Séance 10 – La gestion des exceptions
### Lab : capturer, lever et créer des exceptions – **CORRIGÉ (version formateur)**

| | |
|---|---|
| **Durée estimée** | 60 min (2 × 30 min) + bonus |
| **Projet fil rouge** | GestParc – étape 7 : exceptions métier |

**Objectifs**
- Capturer des exceptions précises avec try / except / else / finally
- Exploiter la hiérarchie des exceptions
- Utiliser assert et raise à bon escient
- Créer des exceptions métier et chaîner les erreurs

**Mode d'emploi**
- Exécuter une cellule : **Maj + Entrée** ; en cas de comportement étrange : *Redémarrer le noyau et tout exécuter*.
- Complétez les cellules marquées `# TODO`, puis exécutez la cellule **✅ Vérification** qui suit : aucune erreur = réponse correcte.
- Difficulté : ⭐ facile · ⭐⭐ intermédiaire · ⭐⭐⭐ avancé · Indices dans les blocs 💡 repliables.

---
## Partie 1 – Capturer des exceptions

> **Rappel**
> ```python
> try:
>     operation_risquee()
> except (KeyError, ValueError) as exc:   # une ou plusieurs exceptions précises
>     traiter(exc)
> else:
>     # seulement si aucune exception
> finally:
>     # toujours exécuté
> ```
> - Hiérarchie : `LookupError` ⊃ `KeyError`, `IndexError` · `ArithmeticError` ⊃ `ZeroDivisionError` · `OSError` ⊃ `FileNotFoundError`

### Exemple guidé : moyenne de mesures robuste

In [1]:
def average_load(raw_values: list[str]) -> float:
    """Moyenne des valeurs numériques ; les valeurs illisibles sont ignorées."""
    values = []
    for raw in raw_values:
        try:
            values.append(float(raw))
        except ValueError:
            print(f"  valeur ignorée : {raw!r}")
    try:
        return sum(values) / len(values)
    except ZeroDivisionError:
        return 0.0


print(average_load(["0.42", "0.91", "erreur", "0.65"]))
print(average_load(["N/A"]))

  valeur ignorée : 'erreur'
0.66
  valeur ignorée : 'N/A'
0.0


### Exercice 1 ⭐ – Conversion sûre d'une saisie

Écrivez `safe_int(text: str, default: int | None = None) -> int | None` qui convertit `text` en entier (espaces tolérés)
et renvoie `default` si la conversion échoue. Puis `safe_ratio(a: float, b: float) -> float | None` qui renvoie `a / b`, ou `None` si `b` vaut 0.

In [2]:
def safe_int(text: str, default: int | None = None) -> int | None:
    """Convertit un texte en entier, ou renvoie default."""
    try:
        return int(text)
    except ValueError:
        return default


def safe_ratio(a: float, b: float) -> float | None:
    """Renvoie a / b, ou None si b est nul."""
    try:
        return a / b
    except ZeroDivisionError:
        return None


print(safe_int(" 42 "), safe_int("4,2"), safe_int("abc", 0), safe_ratio(3, 4), safe_ratio(1, 0))

42 None 0 0.75 None


<details><summary>💡 Indice</summary>

Ne capturez que `ValueError` dans `safe_int` et `ZeroDivisionError` dans `safe_ratio`.

</details>

In [3]:
# ✅ Vérification
assert safe_int(" 42 ") == 42 and safe_int("4,2") is None and safe_int("abc", 0) == 0
assert safe_ratio(3, 4) == 0.75 and safe_ratio(1, 0) is None
print("✅ Exercice 1 réussi")

✅ Exercice 1 réussi


### Exercice 2 ⭐⭐ – Charger une configuration en collectant les erreurs

Écrivez `load_settings(raw: dict[str, str], schema: dict[str, type]) -> tuple[dict, list[str]]` :
pour chaque clé du `schema` (clé → type attendu : `int`, `float` ou `str`), lisez `raw[clé]` et convertissez-la.
- clé absente → erreur `"<clé> : absent"` ;
- conversion impossible → erreur `"<clé> : valeur invalide '<valeur>'"`.

La fonction renvoie le dictionnaire des valeurs valides et la liste des erreurs (dans l'ordre du schéma).

In [4]:
def load_settings(raw: dict[str, str], schema: dict[str, type]) -> tuple[dict, list[str]]:
    """Convertit une configuration texte selon un schéma ; renvoie (valeurs, erreurs)."""
    settings: dict = {}
    errors: list[str] = []
    for key, expected in schema.items():
        try:
            settings[key] = expected(raw[key])
        except KeyError:
            errors.append(f"{key} : absent")
        except ValueError:
            errors.append(f"{key} : valeur invalide {raw[key]!r}")
    return settings, errors


raw = {"host": "srv-db-01", "port": "5432", "timeout": "2,5", "pool_size": "10"}
schema = {"host": str, "port": int, "timeout": float, "pool_size": int, "ssl_mode": str}
settings, errors = load_settings(raw, schema)
print(settings, errors)

{'host': 'srv-db-01', 'port': 5432, 'pool_size': 10} ["timeout : valeur invalide '2,5'", 'ssl_mode : absent']


<details><summary>💡 Indice</summary>

Les types sont des fonctions de conversion : `expected(raw[key])` vaut `int("5432")` quand `expected` est `int`. Deux `except` distincts.

</details>

In [5]:
# ✅ Vérification
assert settings == {"host": "srv-db-01", "port": 5432, "pool_size": 10}
assert errors == ["timeout : valeur invalide '2,5'", "ssl_mode : absent"]
assert load_settings({}, {}) == ({}, [])
print("✅ Exercice 2 réussi")

✅ Exercice 2 réussi


### Exercice 3 ⭐⭐ – else et finally autour d'une connexion

La classe `FakeConnection` simule une connexion à une base de données. Écrivez `run_query(conn, query) -> str | None` qui :
1. ouvre la connexion (`conn.open()`) ;
2. exécute la requête (`conn.execute(query)`, qui peut lever `TimeoutError` ou `PermissionError`) ;
3. en cas de `TimeoutError`, ajoute `"timeout"` au journal et renvoie `None` ;
4. **si tout s'est bien passé**, ajoute `"ok"` au journal (`else`) et renvoie le résultat ;
5. **dans tous les cas**, ferme la connexion (`finally`). `PermissionError` n'est pas capturée : elle remonte à l'appelant.

In [6]:
class FakeConnection:
    """Connexion simulée (fournie)."""

    def __init__(self) -> None:
        self.log: list[str] = []
        self.is_open = False

    def open(self) -> None:
        self.is_open = True
        self.log.append("open")

    def execute(self, query: str) -> str:
        if "slow" in query:
            raise TimeoutError("requête trop longue")
        if query.startswith("DROP"):
            raise PermissionError("droits insuffisants")
        return f"résultat de {query}"

    def close(self) -> None:
        self.is_open = False
        self.log.append("close")


def run_query(conn: FakeConnection, query: str) -> str | None:
    """Exécute une requête en garantissant la fermeture de la connexion."""
    conn.open()
    try:
        result = conn.execute(query)
    except TimeoutError:
        conn.log.append("timeout")
        return None
    else:
        conn.log.append("ok")
        return result
    finally:
        conn.close()


c = FakeConnection()
print(run_query(c, "SELECT 1"), run_query(c, "SELECT slow"), c.log)

résultat de SELECT 1 None ['open', 'ok', 'close', 'open', 'timeout', 'close']


<details><summary>💡 Indice</summary>

Le `finally` s'exécute même après un `return` dans `except` ou `else`, et même si l'exception n'est pas capturée.

</details>

In [7]:
# ✅ Vérification
c = FakeConnection()
assert run_query(c, "SELECT 1") == "résultat de SELECT 1" and c.log == ["open", "ok", "close"]
c = FakeConnection()
assert run_query(c, "SELECT slow") is None and c.log == ["open", "timeout", "close"]
c = FakeConnection()
try:
    run_query(c, "DROP TABLE users")
    raise AssertionError("PermissionError aurait dû remonter")
except PermissionError:
    pass
assert c.log == ["open", "close"] and not c.is_open
print("✅ Exercice 3 réussi")

✅ Exercice 3 réussi


### Exercice 4 ⭐⭐ – Capturer par famille d'exceptions

La fonction `lookup(data, path)` suit un chemin de clés et d'indices dans une structure imbriquée (fournie).
Écrivez `safe_lookup(data, path, default=None)` qui renvoie `default` en cas de clé **ou** d'indice introuvable,
en capturant **une seule** famille d'exceptions. Les autres erreurs (par exemple `TypeError`) doivent remonter.

Puis classez dans `families` le nom de la **classe mère directe** (`type(exc).__mro__[1].__name__`) de chaque exception levée par les opérations de `operations`.

In [8]:
def lookup(data, path: list):
    """Suit un chemin de clés / indices (fourni)."""
    for step in path:
        data = data[step]
    return data


def safe_lookup(data, path: list, default=None):
    """Comme lookup, mais renvoie default si une clé ou un indice est absent."""
    try:
        return lookup(data, path)
    except LookupError:
        return default


infra = {"sites": [{"name": "Tunis", "servers": ["srv-web-01", "srv-db-01"]}, {"name": "Sfax", "servers": []}]}
print(safe_lookup(infra, ["sites", 0, "servers", 1]), safe_lookup(infra, ["sites", 1, "servers", 0], "aucun"))

operations = [lambda: 1 / 0, lambda: [1][5], lambda: {}["x"], lambda: int("x"), lambda: open("/introuvable.txt")]
families = []
for operation in operations:
    try:
        operation()
    except Exception as exc:
        families.append(type(exc).__mro__[1].__name__)
print(families)

srv-db-01 aucun
['ArithmeticError', 'LookupError', 'LookupError', 'Exception', 'OSError']


<details><summary>💡 Indice</summary>

`except LookupError` capture `KeyError` et `IndexError`. Indicer une liste par une chaîne lève `TypeError`.

</details>

In [9]:
# ✅ Vérification
assert safe_lookup(infra, ["sites", 0, "servers", 1]) == "srv-db-01"
assert safe_lookup(infra, ["sites", 9]) is None and safe_lookup(infra, ["region"], "?") == "?"
try:
    safe_lookup(infra, ["sites", "zero"])
    raise AssertionError("TypeError aurait dû remonter")
except TypeError:
    pass
assert families == ["ArithmeticError", "LookupError", "LookupError", "Exception", "OSError"]
print("✅ Exercice 4 réussi")

✅ Exercice 4 réussi


---
## Partie 2 – Assertions, raise et exceptions personnalisées

> **Rappel**
> - `assert condition, "message"` : invariants internes (désactivables avec `python -O`)
> - `raise ValueError("message précis")` · `raise` seul dans un `except` relance l'exception courante
> - `class MonErreur(Exception): ...` · une base métier commune · attributs supplémentaires via `__init__` + `super().__init__(message)`
> - `raise NouvelleErreur(...) from exc` conserve la cause (`__cause__`)

### Exemple guidé : un quota de stockage

In [10]:
class QuotaExceededError(Exception):
    """Le quota de stockage d'un utilisateur est dépassé."""

    def __init__(self, user: str, requested_gb: float, available_gb: float) -> None:
        super().__init__(f"{user} : {requested_gb} Go demandés, {available_gb} Go disponibles")
        self.user = user
        self.missing_gb = requested_gb - available_gb


def upload(user: str, size_gb: float, used_gb: float, quota_gb: float = 50) -> float:
    if size_gb <= 0:
        raise ValueError(f"taille invalide : {size_gb}")
    available = quota_gb - used_gb
    if size_gb > available:
        raise QuotaExceededError(user, size_gb, available)
    return used_gb + size_gb


for size in [5, 20, -1]:
    try:
        print("Nouvel espace utilisé :", upload("amira", size, used_gb=40))
    except QuotaExceededError as exc:
        print("Refus :", exc, f"(manque {exc.missing_gb} Go)")
    except ValueError as exc:
        print("Erreur :", exc)

Nouvel espace utilisé : 45
Refus : amira : 20 Go demandés, 10 Go disponibles (manque 10 Go)
Erreur : taille invalide : -1


### Exercice 5 ⭐ – Assertions de préconditions

Ajoutez des **assertions** à `split_bandwidth(total_mbps, weights)` qui répartit une bande passante proportionnellement à des poids :
- `total_mbps` est strictement positif ;
- `weights` n'est pas vide et tous les poids sont positifs ou nuls ;
- la somme des poids est strictement positive.
Chaque assertion a un message explicite.

In [11]:
def split_bandwidth(total_mbps: float, weights: dict[str, float]) -> dict[str, float]:
    """Répartit une bande passante proportionnellement aux poids."""
    assert total_mbps > 0, f"bande passante invalide : {total_mbps}"
    assert weights, "aucun poids fourni"
    assert all([w >= 0 for w in weights.values()]), f"poids négatif : {weights}"
    total_weight = sum(weights.values())
    assert total_weight > 0, "la somme des poids est nulle"
    return {name: round(total_mbps * w / total_weight, 1) for name, w in weights.items()}


print(split_bandwidth(1000, {"voip": 3, "web": 5, "backup": 2}))

{'voip': 300.0, 'web': 500.0, 'backup': 200.0}


<details><summary>💡 Indice</summary>

`all([...])` vaut `True` si toutes les conditions de la liste sont vraies.

</details>

In [12]:
# ✅ Vérification
assert split_bandwidth(1000, {"voip": 3, "web": 5, "backup": 2}) == {"voip": 300.0, "web": 500.0, "backup": 200.0}
for args in [(0, {"a": 1}), (100, {}), (100, {"a": -1, "b": 2}), (100, {"a": 0})]:
    try:
        split_bandwidth(*args)
        raise RuntimeError(f"assertion attendue pour {args}")
    except AssertionError as exc:
        assert str(exc), "chaque assertion doit avoir un message"
print("✅ Exercice 5 réussi")

✅ Exercice 5 réussi


### Exercice 6 ⭐⭐ – raise dans les propriétés

Reprenez l'onduleur de S7 : au lieu de **ramener** la charge dans [0, 100], le setter doit maintenant **lever** :
- `TypeError` si la valeur n'est ni un `int` ni un `float` (attention : `bool` est une sous-classe d'`int` et doit être refusé) ;
- `ValueError` si la valeur est hors de [0, 100].

`discharge(minutes)` lève `ValueError` si `minutes` est négatif, et ne descend pas sous 0 % (pas d'erreur dans ce cas).

In [13]:
class UPS:
    """Onduleur de salle serveur."""

    def __init__(self, name: str, charge: float = 100.0) -> None:
        self.name = name
        self.charge = charge

    @property
    def charge(self) -> float:
        return self._charge

    @charge.setter
    def charge(self, value: float) -> None:
        if isinstance(value, bool) or not isinstance(value, (int, float)):
            raise TypeError(f"charge numérique attendue, {type(value).__name__} reçu")
        if not 0 <= value <= 100:
            raise ValueError(f"charge hors limites : {value}")
        self._charge = float(value)

    def discharge(self, minutes: int, rate: float = 1.5) -> None:
        if minutes < 0:
            raise ValueError(f"durée négative : {minutes}")
        self.charge = max(0.0, self.charge - minutes * rate)


ups = UPS("ups-a")
ups.discharge(80)
print(ups.charge)

0.0


<details><summary>💡 Indice</summary>

Testez le type **avant** de comparer la valeur : `"50" <= 100` lèverait lui-même une `TypeError` peu explicite.

</details>

In [14]:
# ✅ Vérification
u = UPS("t", 50)
u.discharge(10)
assert u.charge == 35.0
u.discharge(100)
assert u.charge == 0.0
for bad, expected in [(120, ValueError), (-1, ValueError), ("50", TypeError), (True, TypeError), (None, TypeError)]:
    try:
        UPS("x", bad)
        raise AssertionError(f"{expected.__name__} attendue pour {bad!r}")
    except expected:
        pass
try:
    u.discharge(-5)
    raise AssertionError("ValueError attendue")
except ValueError:
    pass
print("✅ Exercice 6 réussi")

✅ Exercice 6 réussi


### Exercice 7 ⭐⭐ – Exceptions d'un pipeline de déploiement

Créez la hiérarchie :
- `DeploymentError(Exception)` : base ;
- `BuildError(DeploymentError)` ;
- `TestFailureError(DeploymentError)` avec un attribut `failed` (nombre de tests en échec) et le message `"3 test(s) en échec"` ;
- `HealthCheckError(DeploymentError)`.

Complétez `deploy(version, build_ok, failed_tests, healthy)` qui lève l'exception adaptée à la **première** étape en échec,
puis `release(...)` qui appelle `deploy` et renvoie : `"v1.4.0 déployée"`, `"v1.4.0 bloquée : <message>"` pour toute `DeploymentError`,
et ajoute `" (rollback)"` uniquement pour une `HealthCheckError`.

In [15]:
class DeploymentError(Exception):
    """Erreur de déploiement."""


class BuildError(DeploymentError):
    """La compilation a échoué."""


class TestFailureError(DeploymentError):
    """Des tests ont échoué."""

    def __init__(self, failed: int) -> None:
        super().__init__(f"{failed} test(s) en échec")
        self.failed = failed


class HealthCheckError(DeploymentError):
    """L'application ne répond pas après déploiement."""


def deploy(version: str, build_ok: bool, failed_tests: int, healthy: bool) -> None:
    """Simule les étapes d'un déploiement."""
    if not build_ok:
        raise BuildError(f"échec de compilation de {version}")
    if failed_tests > 0:
        raise TestFailureError(failed_tests)
    if not healthy:
        raise HealthCheckError("le contrôle /health ne répond pas")


def release(version: str, build_ok: bool = True, failed_tests: int = 0, healthy: bool = True) -> str:
    """Déploie et renvoie un compte rendu."""
    try:
        deploy(version, build_ok, failed_tests, healthy)
    except HealthCheckError as exc:
        return f"{version} bloquée : {exc} (rollback)"
    except DeploymentError as exc:
        return f"{version} bloquée : {exc}"
    return f"{version} déployée"


print(release("v1.4.0"), release("v1.4.1", failed_tests=3), release("v1.4.2", healthy=False), sep="\n")

v1.4.0 déployée
v1.4.1 bloquée : 3 test(s) en échec
v1.4.2 bloquée : le contrôle /health ne répond pas (rollback)


<details><summary>💡 Indice</summary>

Ordre des `except` : la classe la plus précise d'abord (`HealthCheckError`), la classe de base ensuite.

</details>

In [16]:
# ✅ Vérification
assert release("v1.4.0") == "v1.4.0 déployée"
assert release("v1.4.1", failed_tests=3) == "v1.4.1 bloquée : 3 test(s) en échec"
assert release("v1.4.2", healthy=False) == "v1.4.2 bloquée : le contrôle /health ne répond pas (rollback)"
assert release("v1.4.3", build_ok=False, failed_tests=2).startswith("v1.4.3 bloquée : échec de compilation")
try:
    deploy("v", True, 2, True)
except TestFailureError as exc:
    assert exc.failed == 2 and isinstance(exc, DeploymentError)
print("✅ Exercice 7 réussi")

✅ Exercice 7 réussi


### Exercice 8 ⭐⭐⭐ – Nouvelles tentatives et raise ... from

Écrivez `retry(operation, attempts=3, retry_on=(ConnectionError,))` :
- appelle `operation()` jusqu'à `attempts` fois ;
- renvoie son résultat dès qu'un appel réussit ;
- recommence seulement pour les exceptions de `retry_on` ; toute autre exception remonte **immédiatement** ;
- si toutes les tentatives échouent, lève `RetryError(f"échec après {attempts} tentative(s)")` **chaînée** à la dernière exception (`from`).

`FlakyService` (fourni) échoue un certain nombre de fois avant de répondre.

In [17]:
class RetryError(Exception):
    """Toutes les tentatives ont échoué."""


class FlakyService:
    """Service instable (fourni) : échoue `failures` fois puis répond."""

    def __init__(self, failures: int, error: type[Exception] = ConnectionError) -> None:
        self.failures = failures
        self.error = error
        self.calls = 0

    def __call__(self) -> str:
        self.calls += 1
        if self.calls <= self.failures:
            raise self.error(f"échec n° {self.calls}")
        return "200 OK"


def retry(operation, attempts: int = 3, retry_on: tuple = (ConnectionError,)):
    """Exécute operation avec plusieurs tentatives."""
    last_error = None
    for _ in range(attempts):
        try:
            return operation()
        except retry_on as exc:
            last_error = exc
    raise RetryError(f"échec après {attempts} tentative(s)") from last_error


print(retry(FlakyService(2)))
try:
    retry(FlakyService(5), attempts=3)
except RetryError as exc:
    print(exc, "| cause :", repr(exc.__cause__))

200 OK
échec après 3 tentative(s) | cause : ConnectionError('échec n° 3')


<details><summary>💡 Indice</summary>

`except retry_on as exc:` fonctionne avec un tuple d'exceptions. Gardez la dernière exception dans une variable pour le `from`.

</details>

In [18]:
# ✅ Vérification
s = FlakyService(2)
assert retry(s) == "200 OK" and s.calls == 3
s = FlakyService(5)
try:
    retry(s, attempts=4)
    raise AssertionError("RetryError attendue")
except RetryError as exc:
    assert str(exc) == "échec après 4 tentative(s)" and s.calls == 4
    assert isinstance(exc.__cause__, ConnectionError) and str(exc.__cause__) == "échec n° 4"
s = FlakyService(1, error=PermissionError)
try:
    retry(s)
    raise AssertionError("PermissionError attendue")
except PermissionError:
    assert s.calls == 1
assert retry(FlakyService(1, TimeoutError), retry_on=(TimeoutError, ConnectionError)) == "200 OK"
print("✅ Exercice 8 réussi")

✅ Exercice 8 réussi


---
### Exercice de synthèse ⭐⭐⭐ – Import robuste d'une liste d'employés

Un export RH contient des lignes `matricule;prénom;nom;service;date_embauche(AAAA-MM-JJ)`.
1. Créez `InvalidRecordError(ValueError)` avec les attributs `line_number` et `reason`, et le message `"ligne 3 : <raison>"`.
2. Écrivez `parse_record(line_number, line) -> dict` qui lève `InvalidRecordError` si :
   - le nombre de champs n'est pas 5 → raison `"5 champs attendus, 4 reçus"` ;
   - le matricule n'est pas `E` suivi de 4 chiffres → `"matricule invalide 'X12'"` ;
   - un champ est vide → `"champ vide"` ;
   - l'année d'embauche n'est pas un entier entre 1990 et 2026 → `"date invalide '...'"` (utilisez `raise ... from` pour une erreur de conversion).
3. Écrivez `import_employees(lines) -> tuple[list[dict], list[str]]` qui ignore les lignes vides, collecte les employés valides et les messages d'erreur, sans jamais s'arrêter.

In [19]:
class InvalidRecordError(ValueError):
    """Ligne d'import invalide."""

    def __init__(self, line_number: int, reason: str) -> None:
        super().__init__(f"ligne {line_number} : {reason}")
        self.line_number = line_number
        self.reason = reason


def parse_record(line_number: int, line: str) -> dict:
    """Analyse une ligne de l'export RH."""
    fields = [f.strip() for f in line.split(";")]
    if len(fields) != 5:
        raise InvalidRecordError(line_number, f"5 champs attendus, {len(fields)} reçus")
    emp_id, first, last, department, hired = fields
    if not all(fields):
        raise InvalidRecordError(line_number, "champ vide")
    if not (len(emp_id) == 5 and emp_id[0] == "E" and emp_id[1:].isdigit()):
        raise InvalidRecordError(line_number, f"matricule invalide {emp_id!r}")
    try:
        year = int(hired[:4])
    except ValueError as exc:
        raise InvalidRecordError(line_number, f"date invalide {hired!r}") from exc
    if not 1990 <= year <= 2026:
        raise InvalidRecordError(line_number, f"date invalide {hired!r}")
    return {"id": emp_id, "first_name": first, "last_name": last, "department": department, "hired": hired}


def import_employees(lines: list[str]) -> tuple[list[dict], list[str]]:
    """Importe les lignes valides et collecte les erreurs."""
    employees, errors = [], []
    for number, line in enumerate(lines, start=1):
        if not line.strip():
            continue
        try:
            employees.append(parse_record(number, line))
        except InvalidRecordError as exc:
            errors.append(str(exc))
    return employees, errors


export = [
    "E0001;Amira;Ben Salah;DSI;2019-03-01",
    "E0002;Karim;Trabelsi;Finance",
    "X12;Sonia;Haddad;DSI;2021-09-15",
    "",
    "E0004;Youssef;;RH;2020-01-06",
    "E0005;Leila;Mansour;DSI;deux-mille",
    "E0006;Hedi;Jaziri;Achats;1985-05-20",
    "E0007;Ines;Ayari;DSI;2024-02-01",
]
employees, errors = import_employees(export)
print([e["id"] for e in employees])
print("\n".join(errors))

['E0001', 'E0007']
ligne 2 : 5 champs attendus, 4 reçus
ligne 3 : matricule invalide 'X12'
ligne 5 : champ vide
ligne 6 : date invalide 'deux-mille'
ligne 7 : date invalide '1985-05-20'


<details><summary>💡 Indice</summary>

Vérifiez les champs vides avant le matricule (l'ordre des contrôles détermine le message). `all(fields)` est faux si un champ est vide.

</details>

In [20]:
# ✅ Vérification
assert [e["id"] for e in employees] == ["E0001", "E0007"]
assert errors == [
    "ligne 2 : 5 champs attendus, 4 reçus",
    "ligne 3 : matricule invalide 'X12'",
    "ligne 5 : champ vide",
    "ligne 6 : date invalide 'deux-mille'",
    "ligne 7 : date invalide '1985-05-20'",
]
try:
    parse_record(6, "E0005;Leila;Mansour;DSI;deux-mille")
except InvalidRecordError as exc:
    assert isinstance(exc, ValueError) and exc.line_number == 6 and isinstance(exc.__cause__, ValueError)
print("✅ Exercice de synthèse réussi")

✅ Exercice de synthèse réussi


---
## Projet fil rouge – étape 7 : exceptions métier

![Hiérarchie des exceptions](img/s10_exceptions.svg)

Jusqu'ici, GestParc signalait les problèmes par des `print` (« Prix refusé ») ou des `False` silencieux. On les remplace par des **exceptions métier** :

| Exception | Bases | Levée par |
|---|---|---|
| `GestParcError` | `Exception` | (base commune) |
| `InvalidEquipmentDataError` | `GestParcError`, `ValueError` | numéro invalide, prix négatif ou non numérique, IP invalide, catégorie inconnue |
| `DuplicateAssetTagError(asset_tag)` | `GestParcError` | `Inventory.add` |
| `EquipmentNotFoundError(asset_tag)` | `GestParcError`, `LookupError` | nouvelle méthode `Inventory.get` |
| `AlreadyAssignedError(asset_tag, owner)` | `GestParcError` | `Equipment.assign` (qui ne renvoie plus de booléen) |

`Inventory.add` lève aussi `TypeError` si on lui passe autre chose qu'un `Equipment`. Nouvelle méthode `Inventory.remove(tag)`.
Les zones à modifier sont marquées `# TODO` dans le code ci-dessous (repris de l'étape 6).

In [21]:
from abc import ABC, abstractmethod


class GestParcError(Exception):
    """Classe de base de toutes les erreurs métier de GestParc."""


class InvalidEquipmentDataError(GestParcError, ValueError):
    """Donnée d'équipement invalide (numéro, prix, adresse IP...)."""


class DuplicateAssetTagError(GestParcError):
    """Numéro d'inventaire déjà présent dans l'inventaire."""

    def __init__(self, asset_tag: str) -> None:
        super().__init__(f"Le numéro {asset_tag} existe déjà dans l'inventaire")
        self.asset_tag = asset_tag


class EquipmentNotFoundError(GestParcError, LookupError):
    """Aucun équipement ne porte ce numéro."""

    def __init__(self, asset_tag: str) -> None:
        super().__init__(f"Équipement introuvable : {asset_tag}")
        self.asset_tag = asset_tag


class AlreadyAssignedError(GestParcError):
    """L'équipement est déjà affecté à quelqu'un."""

    def __init__(self, asset_tag: str, owner: str) -> None:
        super().__init__(f"{asset_tag} est déjà affecté à {owner}")
        self.asset_tag = asset_tag
        self.owner = owner


class NetworkMixin:
    """Ajoute une adresse IP gérée à un équipement réseau."""

    _ip_address: str | None = None

    @staticmethod
    def is_valid_ip(ip: str) -> bool:
        """Vérifie le format d'une adresse IPv4."""
        parts = ip.split(".")
        return len(parts) == 4 and all([p.isdigit() and 0 <= int(p) <= 255 for p in parts])

    @property
    def ip_address(self) -> str | None:
        return self._ip_address

    @ip_address.setter
    def ip_address(self, value: str | None) -> None:
        if value is not None and not self.is_valid_ip(value):
            raise InvalidEquipmentDataError(f"Adresse IP invalide : {value!r}")
        self._ip_address = value


class Equipment(ABC):
    """Équipement du parc informatique (classe abstraite)."""

    CATEGORY = "equipment"
    PREFIX = "EQ"
    count = 0

    def __init__(self, asset_tag: str, brand: str, model: str, purchase_year: int,
                 price: float, assigned_to: str | None = None) -> None:
        if not Equipment.is_valid_tag(asset_tag.upper()):
            raise InvalidEquipmentDataError(f"Numéro d'inventaire invalide : {asset_tag!r} (format attendu : LT-001)")
        self.asset_tag = asset_tag.upper()
        self.brand = brand
        self.model = model
        self.purchase_year = purchase_year
        self._price = 0.0
        self.price = price
        self._assigned_to = assigned_to
        Equipment.count += 1

    # --- Méthodes abstraites : chaque sous-classe DOIT les définir
    @abstractmethod
    def maintenance_interval_months(self) -> int:
        """Périodicité de maintenance préventive, en mois."""

    @abstractmethod
    def specs(self) -> str:
        """Caractéristiques techniques propres au type d'équipement."""

    # --- Méthodes communes
    @property
    def category(self) -> str:
        return self.CATEGORY

    @staticmethod
    def is_valid_tag(tag: str) -> bool:
        return len(tag) == 6 and tag[:2].isalpha() and tag[2] == "-" and tag[3:].isdigit()

    def has_expected_prefix(self) -> bool:
        return self.asset_tag.startswith(self.PREFIX + "-")

    @property
    def price(self) -> float:
        return self._price

    @price.setter
    def price(self, value: float) -> None:
        try:
            amount = float(value)
        except (TypeError, ValueError) as exc:
            raise InvalidEquipmentDataError(f"Prix non numérique pour {self.asset_tag} : {value!r}") from exc
        if amount < 0:
            raise InvalidEquipmentDataError(f"Prix négatif pour {self.asset_tag} : {amount}")
        self._price = amount

    @property
    def assigned_to(self) -> str | None:
        return self._assigned_to

    def age(self, current_year: int) -> int:
        return current_year - self.purchase_year

    def is_available(self) -> bool:
        return self._assigned_to is None

    def assign(self, employee: str) -> None:
        """Affecte l'équipement ; lève AlreadyAssignedError s'il est déjà affecté."""
        if not self.is_available():
            raise AlreadyAssignedError(self.asset_tag, self._assigned_to)
        self._assigned_to = employee

    def unassign(self) -> None:
        self._assigned_to = None

    def to_dict(self) -> dict:
        return {"asset_tag": self.asset_tag, "category": self.category, "brand": self.brand,
                "model": self.model, "purchase_year": self.purchase_year, "price": self.price,
                "assigned_to": self.assigned_to}

    def __str__(self) -> str:
        name = f"{self.brand} {self.model}"
        return (f"{self.asset_tag:<7} | {self.category:<8} | {name:<25} | "
                f"{self.price:>10,.2f} DT | {self.assigned_to or '-'}")

    def __repr__(self) -> str:
        return f"{type(self).__name__}({self.asset_tag!r}, {self.brand!r}, {self.model!r})"

    def __eq__(self, other: object) -> bool:
        return isinstance(other, Equipment) and self.asset_tag == other.asset_tag

    def __lt__(self, other: "Equipment") -> bool:
        return self.asset_tag < other.asset_tag


class Laptop(Equipment):
    """Ordinateur portable."""

    CATEGORY = "laptop"
    PREFIX = "LT"

    def __init__(self, asset_tag: str, brand: str, model: str, purchase_year: int, price: float,
                 assigned_to: str | None = None, *, ram_gb: int = 16, screen_inches: float = 14.0) -> None:
        super().__init__(asset_tag, brand, model, purchase_year, price, assigned_to)
        self.ram_gb = ram_gb
        self.screen_inches = screen_inches

    def maintenance_interval_months(self) -> int:
        return 24

    def specs(self) -> str:
        return f"{self.ram_gb} Go RAM, écran {self.screen_inches}\""

    def to_dict(self) -> dict:
        data = super().to_dict()
        data.update(ram_gb=self.ram_gb, screen_inches=self.screen_inches)
        return data


class Server(NetworkMixin, Equipment):
    """Serveur physique."""

    CATEGORY = "server"
    PREFIX = "SV"

    def __init__(self, asset_tag: str, brand: str, model: str, purchase_year: int, price: float,
                 assigned_to: str | None = None, *, cpu_count: int = 16, ram_gb: int = 64,
                 rack: str = "R1", ip_address: str | None = None) -> None:
        super().__init__(asset_tag, brand, model, purchase_year, price, assigned_to)
        self.cpu_count = cpu_count
        self.ram_gb = ram_gb
        self.rack = rack
        self.ip_address = ip_address

    def maintenance_interval_months(self) -> int:
        return 6

    def specs(self) -> str:
        return f"{self.cpu_count} cœurs, {self.ram_gb} Go RAM, baie {self.rack}, IP {self.ip_address or '-'}"

    def to_dict(self) -> dict:
        data = super().to_dict()
        data.update(cpu_count=self.cpu_count, ram_gb=self.ram_gb, rack=self.rack, ip_address=self.ip_address)
        return data


class Printer(NetworkMixin, Equipment):
    """Imprimante réseau."""

    CATEGORY = "printer"
    PREFIX = "PR"

    def __init__(self, asset_tag: str, brand: str, model: str, purchase_year: int, price: float,
                 assigned_to: str | None = None, *, is_color: bool = False, pages_printed: int = 0,
                 ip_address: str | None = None) -> None:
        super().__init__(asset_tag, brand, model, purchase_year, price, assigned_to)
        self.is_color = is_color
        self.pages_printed = pages_printed
        self.ip_address = ip_address

    def maintenance_interval_months(self) -> int:
        # Une imprimante très sollicitée est entretenue plus souvent
        return 3 if self.pages_printed > 50_000 else 12

    def specs(self) -> str:
        return f"{'couleur' if self.is_color else 'monochrome'}, {self.pages_printed:,} pages, IP {self.ip_address or '-'}"

    def to_dict(self) -> dict:
        data = super().to_dict()
        data.update(is_color=self.is_color, pages_printed=self.pages_printed, ip_address=self.ip_address)
        return data


class NetworkDevice(NetworkMixin, Equipment):
    """Équipement réseau (switch, routeur, point d'accès)."""

    CATEGORY = "network"
    PREFIX = "SW"

    def __init__(self, asset_tag: str, brand: str, model: str, purchase_year: int, price: float,
                 assigned_to: str | None = None, *, port_count: int = 24,
                 ip_address: str | None = None) -> None:
        super().__init__(asset_tag, brand, model, purchase_year, price, assigned_to)
        self.port_count = port_count
        self.ip_address = ip_address

    def maintenance_interval_months(self) -> int:
        return 12

    def specs(self) -> str:
        return f"{self.port_count} ports, IP {self.ip_address or '-'}"

    def to_dict(self) -> dict:
        data = super().to_dict()
        data.update(port_count=self.port_count, ip_address=self.ip_address)
        return data


EQUIPMENT_TYPES: dict[str, type[Equipment]] = {
    cls.CATEGORY: cls for cls in (Laptop, Server, Printer, NetworkDevice)
}


def equipment_from_dict(data: dict) -> Equipment:
    """Fabrique : crée l'objet de la bonne sous-classe à partir d'un dictionnaire."""
    try:
        cls = EQUIPMENT_TYPES[data["category"]]
    except KeyError as exc:
        raise InvalidEquipmentDataError(f"Catégorie inconnue : {data.get('category')!r}") from exc
    common = ["asset_tag", "brand", "model", "purchase_year", "price", "assigned_to"]
    specific = {k: v for k, v in data.items() if k not in common and k != "category"}
    return cls(data["asset_tag"], data["brand"], data["model"], data["purchase_year"],
               data["price"], data.get("assigned_to"), **specific)


class Employee:
    """Employé pouvant recevoir des équipements."""

    def __init__(self, first_name: str, last_name: str, department: str) -> None:
        self.first_name = first_name
        self.last_name = last_name
        self.department = department

    @property
    def full_name(self) -> str:
        return f"{self.first_name} {self.last_name}"

    def __str__(self) -> str:
        return f"{self.full_name} ({self.department})"


class Inventory:
    """Inventaire du parc informatique."""

    def __init__(self) -> None:
        self._items: list[Equipment] = []

    def add(self, equipment: Equipment) -> None:
        """Ajoute un équipement ; lève TypeError ou DuplicateAssetTagError."""
        if not isinstance(equipment, Equipment):
            raise TypeError(f"Equipment attendu, {type(equipment).__name__} reçu")
        if equipment in self._items:
            raise DuplicateAssetTagError(equipment.asset_tag)
        self._items.append(equipment)

    def find(self, asset_tag: str) -> Equipment | None:
        tag = asset_tag.upper()
        for equipment in self._items:
            if equipment.asset_tag == tag:
                return equipment
        return None

    def get(self, asset_tag: str) -> Equipment:
        """Comme find, mais lève EquipmentNotFoundError si le numéro est inconnu."""
        equipment = self.find(asset_tag)
        if equipment is None:
            raise EquipmentNotFoundError(asset_tag.upper())
        return equipment

    def assign(self, asset_tag: str, employee: Employee) -> None:
        self.get(asset_tag).assign(employee.full_name)

    def remove(self, asset_tag: str) -> Equipment:
        equipment = self.get(asset_tag)
        self._items.remove(equipment)
        return equipment

    def by_type(self, cls: type) -> list[Equipment]:
        """Équipements instances d'une classe donnée (sous-classes comprises)."""
        return [e for e in self._items if isinstance(e, cls)]

    def stats(self) -> dict[str, int]:
        result: dict[str, int] = {}
        for equipment in self._items:
            result[equipment.category] = result.get(equipment.category, 0) + 1
        return result

    def total_value(self, category: str | None = None) -> float:
        return sum([e.price for e in self._items if category is None or e.category == category])

    def available(self) -> list[Equipment]:
        return [e for e in self._items if e.is_available()]

    def older_than(self, years: int, current_year: int) -> list[Equipment]:
        return [e for e in self._items if e.age(current_year) >= years]

    def maintenance_plan(self) -> list[tuple[str, int]]:
        """Polymorphisme : chaque équipement donne sa propre périodicité."""
        return sorted([(e.asset_tag, e.maintenance_interval_months()) for e in self._items],
                      key=lambda item: item[1])

    def ip_plan(self) -> dict[str, str | None]:
        """Adresses IP des équipements réseau (ceux qui ont le mixin)."""
        return {e.asset_tag: e.ip_address for e in self._items if isinstance(e, NetworkMixin)}

    def __len__(self) -> int:
        return len(self._items)

    def __contains__(self, item: object) -> bool:
        if isinstance(item, str):
            return self.find(item) is not None
        return item in self._items

    def __getitem__(self, index: int) -> Equipment:
        return self._items[index]


def demo_inventory() -> Inventory:
    """Inventaire de démonstration."""
    inventory = Inventory()
    inventory.add(Server("SV-001", "HPE", "ProLiant DL380", 2019, 18500, cpu_count=32, ram_gb=256,
                         rack="R2", ip_address="10.0.0.10"))
    inventory.add(Laptop("LT-001", "Dell", "Latitude 5440", 2023, 3200, "Amira Ben Salah", ram_gb=32))
    inventory.add(Printer("PR-001", "HP", "LaserJet M404", 2021, 950, pages_printed=62_000,
                          ip_address="10.0.2.50"))
    inventory.add(NetworkDevice("SW-001", "Cisco", "Catalyst 9200", 2020, 4100, port_count=48,
                                ip_address="10.0.0.2"))
    return inventory


inventory = demo_inventory()
attempts = [
    lambda: Laptop("LT-1", "Dell", "XPS", 2024, 4000),
    lambda: Laptop("LT-002", "Dell", "XPS", 2024, "quatre mille"),
    lambda: Server("SV-002", "Dell", "R760", 2024, 21000, ip_address="10.0.0.300"),
    lambda: inventory.add(Laptop("LT-001", "Asus", "B5", 2024, 2500)),
    lambda: inventory.get("ZZ-404"),
    lambda: inventory.assign("LT-001", Employee("Karim", "Trabelsi", "Finance")),
    lambda: equipment_from_dict({"category": "tablet", "asset_tag": "TB-001", "brand": "A", "model": "B",
                                 "purchase_year": 2024, "price": 1}),
]
for attempt in attempts:
    try:
        attempt()
    except GestParcError as exc:
        print(f"{type(exc).__name__:<26} {exc}")

InvalidEquipmentDataError  Numéro d'inventaire invalide : 'LT-1' (format attendu : LT-001)
InvalidEquipmentDataError  Prix non numérique pour LT-002 : 'quatre mille'
InvalidEquipmentDataError  Adresse IP invalide : '10.0.0.300'
DuplicateAssetTagError     Le numéro LT-001 existe déjà dans l'inventaire
EquipmentNotFoundError     Équipement introuvable : ZZ-404
AlreadyAssignedError       LT-001 est déjà affecté à Amira Ben Salah
InvalidEquipmentDataError  Catégorie inconnue : 'tablet'


In [22]:
# ✅ Vérification
def raises(exc_type, func):
    try:
        func()
    except exc_type as exc:
        return exc
    raise AssertionError(f"{exc_type.__name__} attendue")


assert issubclass(InvalidEquipmentDataError, ValueError) and issubclass(EquipmentNotFoundError, LookupError)
assert all(issubclass(c, GestParcError) for c in [InvalidEquipmentDataError, DuplicateAssetTagError,
                                                    EquipmentNotFoundError, AlreadyAssignedError])
raises(InvalidEquipmentDataError, lambda: Laptop("LT-1", "D", "X", 2024, 1))
e = raises(InvalidEquipmentDataError, lambda: Laptop("LT-002", "D", "X", 2024, "abc"))
assert isinstance(e.__cause__, ValueError)
raises(InvalidEquipmentDataError, lambda: Laptop("LT-002", "D", "X", 2024, -1))
raises(InvalidEquipmentDataError, lambda: Printer("PR-002", "HP", "M", 2024, 1, ip_address="1.2.3"))
inv = demo_inventory()
assert raises(DuplicateAssetTagError, lambda: inv.add(Laptop("LT-001", "A", "B", 2024, 1))).asset_tag == "LT-001"
raises(TypeError, lambda: inv.add("LT-005"))
assert raises(EquipmentNotFoundError, lambda: inv.get("zz-404")).asset_tag == "ZZ-404"
err = raises(AlreadyAssignedError, lambda: inv.assign("lt-001", Employee("K", "T", "F")))
assert err.owner == "Amira Ben Salah" and str(err) == "LT-001 est déjà affecté à Amira Ben Salah"
inv.assign("SV-001", Employee("Équipe", "Infra", "DSI"))
assert inv.get("sv-001").assigned_to == "Équipe Infra"
assert inv.remove("PR-001").asset_tag == "PR-001" and len(inv) == 3
raises(InvalidEquipmentDataError, lambda: equipment_from_dict({"category": "tablet"}))
print("✅ Étape 7 du projet GestParc validée")

✅ Étape 7 du projet GestParc validée


✔️ Version de référence : `scripts/gestparc/main.py` — le menu intercepte `GestParcError` et `ValueError` dans **un seul** bloc `try` : essayez de saisir un prix non numérique, un numéro déjà utilisé, un numéro inconnu…

**Prochaine étape (S12)** : après le TP de S11, on rendra l'inventaire **persistant** (fichiers texte, dates de garantie, analyse de journaux).

---
## Pour aller plus loin (bonus)

1. **`ExceptionGroup` et `except*`** (Python 3.11+) : faites lever par `import_employees` un `ExceptionGroup` regroupant toutes les erreurs, et traitez-les avec `except* InvalidRecordError`.
2. **`add_note`** (Python 3.11+) : ajoutez une note de contexte à une exception (`exc.add_note("import du 21/10")`) puis relancez-la.
3. **`contextlib.suppress`** : réécrivez un `try/except KeyError: pass` avec `with suppress(KeyError):` (l'instruction `with` sera détaillée en S12).

In [23]:
try:
    raise ExceptionGroup("import RH", [InvalidRecordError(2, "5 champs attendus"), InvalidRecordError(5, "champ vide")])
except* InvalidRecordError as group:
    for exc in group.exceptions:
        print("→", exc)

→ ligne 2 : 5 champs attendus
→ ligne 5 : champ vide
